# Stage 3 — Improved Ensemble

**PMLDL 2026 project — Minimal Requirement 3.** Three model families blended,
Optuna over both hyperparameters and blend weights, and a one-parameter
allocation rule.

## Attribution

| Borrowed | Source |
| --- | --- |
| Fixed-weight ensemble across ElasticNet / XGBoost / LightGBM; metric-anchored position sizing; the observation that a Ridge meta-learner overfits small CV folds | 100th place write-up (competition write-ups page) |
| Single-LightGBM pipeline, lag/rolling context, Optuna tuned on mean Spearman | 61st place notebook, <https://www.kaggle.com/code/rafanikitas/hull-eda-training-pipeline> |
| Domain signal family — mean reversion, volatility spreads, vol targeting | 4th place write-up (competition write-ups page) |
| Cross terms `U1`, `U2` | Hull starter notebook |

Feature engineering is inherited unchanged from Stage 2 so that the measured
improvement is attributable to the ensemble and the allocation rule, not to a
different feature set. No author names, usernames or team identifiers appear
anywhere in this project. Shared infrastructure is documented in `INTERFACE.md`;
seed fixed at 42; folds are the same `get_folds()` defaults used by every other
notebook.

## What changes against Stage 2, and why

Three changes, each targeting a specific weakness visible in the earlier runs.

**1. Ensemble across families.** Stage 2 is a single LightGBM. Averaging
predictors whose errors are not perfectly correlated reduces the variance of the
blend without raising its bias — and the families here fail differently by
construction: gradient boosting on histogram splits (LightGBM), ordered boosting
with a different regularisation path (CatBoost), and a penalised *linear* model
(Ridge/ElasticNet) that cannot represent interactions at all. In a regime this
noisy, prediction variance is the dominant error term, so this is where the
cheap gain is.

**2. Blend weights tuned, not assumed.** The 100th place author used fixed
0.30/0.35/0.35 weights and reported that a Ridge meta-learner overfit their
~135-row folds. Our folds are considerably larger, so we search the weights
directly — but over a **three-parameter simplex**, not a stacked meta-model.
Three parameters fitted on several hundred out-of-fold rows is a very different
proposition from a meta-learner with one coefficient per feature.

**3. Naive allocation instead of the binary policy.** This is the change most
likely to move the score, and the reasoning is about the metric rather than the
model. The binary policy from the 61st place solution sits at `w = 0` on any
negative prediction, so a strategy that is wrong about direction slightly more
than half the time spends much of its life out of the market — which the metric
punishes through the quadratic underperformance term against buy-and-hold. The
naive rule

```
position = clip(1.0 + k * prediction, 0.0, 2.0)
```

is anchored at the passive `w = 1` benchmark and tilts away from it in
proportion to the signal. It inherits the benchmark's return by default and
risks only the tilt. One parameter, no fitted function, nothing to overfit
beyond a single scalar.

## Guarding the comparison

Tuning hyperparameters *and* blend weights *and* `k` on the same folds is a real
selection-bias risk. Three mitigations:

* hyperparameters are tuned per family on fold-level validation; blend weights
  and `k` are then fitted on **out-of-fold predictions only**, never on rows a
  model saw in training;
* `k` is searched over a coarse grid and its sensitivity curve is printed — a
  sharp optimum would be evidence of overfitting, a flat one evidence that the
  choice is safe;
* the held-out 180-row public block is scored **once**, at the end, and nothing
  is selected on it.

In [1]:
# ============================ SETUP ============================
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from src import *          # shared interface, see INTERFACE.md

optuna.logging.set_verbosity(optuna.logging.WARNING)
set_seed()                 # SEED = 42
print("seed:", SEED, "| target:", TARGET)

QUICK = False
N_TRIALS_MODEL = 10 if QUICK else 60    # per family
N_TRIALS_BLEND = 50 if QUICK else 300
TIMEOUT_MODEL = 240 if QUICK else 1800

seed: 42 | target: market_forward_excess_returns


## 1. Data and features — inherited from Stage 2

Identical calls, so any difference in results comes from the model, not the
inputs.

In [2]:
hull = load_dataset()

LAG_ROLL_COLUMNS = ["M4", "V13", "S5", "S2", "D2", "E19", "P7", "P6",
                    "P3", "P13", "P4", "P5", "M2", "V5"]
LAG_ROLL_COLUMNS = [c for c in LAG_ROLL_COLUMNS if c in hull.full.columns]

feat_df, FEATURES = build_features(
    hull.full, lag_roll_columns=LAG_ROLL_COLUMNS,
    price_features=True, cross_terms=True,
)

cut = feat_df[DATE_COL].max() - PUBLIC_TEST_SIZE
train_df = feat_df[feat_df[DATE_COL] <= cut].reset_index(drop=True)
public_df = feat_df[feat_df[DATE_COL] > cut].reset_index(drop=True)
train_df, public_df = impute(train_df, public_df, columns=FEATURES)

assert not set(FEATURES) & set(LOOKAHEAD_COLS), "look-ahead column in feature list"
assert train_df[FEATURES].isna().sum().sum() == 0

X, y = train_df[FEATURES], train_df[TARGET]
folds = get_folds(train_df)          # defaults — same folds as every notebook
assert_no_leakage(folds)
display(describe_folds(train_df, folds))
print(f"{len(FEATURES)} features | train {train_df.shape} | public {public_df.shape}")

,fold,n_train,n_val,train_end_date_id,val_start_date_id,val_end_date_id,gap_days
0,0,1571,1552,2576,2598,4149,22
1,1,3143,1552,4148,4170,5721,22
2,2,4715,1552,5720,5742,7293,22
3,3,6287,1554,7292,7314,8867,22


335 features | train (7862, 339) | public (180, 339)


## 2. Per-family fitting

Each family gets a fit function with the same signature, so the tuning and
out-of-fold machinery below is shared. The linear model is the only one that
needs scaling, and its scaler is fitted on training rows only, inside the fold.

In [3]:
def fit_lgbm(X_tr, y_tr, X_va, y_va, params):
    model = lgb.LGBMRegressor(**{**params, "random_state": SEED, "verbosity": -1})
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="rmse",
              callbacks=[lgb.early_stopping(200, verbose=False),
                         lgb.log_evaluation(-1)])
    return model.predict(X_va), model


def fit_catboost(X_tr, y_tr, X_va, y_va, params):
    model = CatBoostRegressor(**{**params, "random_seed": SEED, "verbose": 0,
                                 "loss_function": "RMSE",
                                 "early_stopping_rounds": 200})
    model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
    return model.predict(X_va), model


def fit_ridge(X_tr, y_tr, X_va, y_va, params):
    scaler = StandardScaler().fit(X_tr)          # train-only statistics
    model = Ridge(alpha=params["alpha"], random_state=SEED)
    model.fit(scaler.transform(X_tr), y_tr)
    return model.predict(scaler.transform(X_va)), (scaler, model)


FITTERS = {"lgbm": fit_lgbm, "catboost": fit_catboost, "ridge": fit_ridge}


def cv_predictions(params, family, feature_list=FEATURES):
    """Out-of-fold predictions and per-fold Spearman on the shared folds."""
    oof = np.full(len(train_df), np.nan)
    scores = []
    for fold in folds:
        X_tr = train_df.iloc[fold.train_idx][feature_list]
        y_tr = y.iloc[fold.train_idx]
        X_va = train_df.iloc[fold.val_idx][feature_list]
        y_va = y.iloc[fold.val_idx]
        pred, _ = FITTERS[family](X_tr, y_tr, X_va, y_va, params)
        oof[fold.val_idx] = pred
        scores.append(spearman_ic(y_va.to_numpy(), pred))
    return oof, float(np.mean(scores)), float(np.std(scores))

## 3. Hyperparameter tuning, one family at a time

All three maximise **mean Spearman rank correlation** across folds, following
the 61st place reasoning: exact magnitudes of a de-meaned, winsorised excess
return are not learnable, ordering is, and ordering is all the allocation rule
consumes. Samplers are seeded, so the searches are reproducible.

In [ ]:
def tune(family, space_fn, n_trials=N_TRIALS_MODEL):
    def objective(trial):
        return cv_predictions(space_fn(trial), family)[1]

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        study_name=f"stage3_{family}",
    )
    study.optimize(objective, n_trials=n_trials, timeout=TIMEOUT_MODEL,
                   show_progress_bar=False)
    print(f"{family:9s} | trials {len(study.trials):3d} | "
          f"best mean IC {study.best_value:+.4f}")
    return study.best_params


def space_lgbm(t):
    return {
        "objective": "regression", "metric": "rmse", "subsample_freq": 1,
        "n_estimators": t.suggest_int("n_estimators", 500, 5000),
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.07, log=True),
        "max_depth": t.suggest_int("max_depth", 4, 12),
        "num_leaves": t.suggest_int("num_leaves", 32, 512),
        "reg_lambda": t.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "reg_alpha": t.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.6, 1.0),
        "subsample": t.suggest_float("subsample", 0.6, 1.0),
    }


def space_catboost(t):
    return {
        "iterations": t.suggest_int("iterations", 300, 3000),
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.10, log=True),
        "depth": t.suggest_int("depth", 4, 10),
        "l2_leaf_reg": t.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
        "subsample": t.suggest_float("subsample", 0.6, 1.0),
        "bootstrap_type": "Bernoulli",
    }


def space_ridge(t):
    return {"alpha": t.suggest_float("alpha", 1e-2, 1e4, log=True)}


BEST = {}
BEST["lgbm"] = {"objective": "regression", "metric": "rmse",
                "subsample_freq": 1, **tune("lgbm", space_lgbm)}
BEST["catboost"] = {"bootstrap_type": "Bernoulli",
                    **tune("catboost", space_catboost)}
BEST["ridge"] = tune("ridge", space_ridge)

c:\Users\bobs\Desktop\hull-tactical-pmldl\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\bobs\Desktop\hull-tactical-pmldl\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\bobs\Desktop\hull-tactical-pmldl\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\bobs\Desktop\hull-tactical-pmldl\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' i

## 4. Out-of-fold predictions

One OOF vector per family. These rows were never seen in training by the model
that predicted them, which is what makes them a legitimate surface for fitting
the blend weights in the next section.

In [ ]:
oof, family_stats = {}, []
for family in ("lgbm", "catboost", "ridge"):
    oof[family], ic_mean, ic_std = cv_predictions(BEST[family], family)
    family_stats.append({"family": family, "ic_mean": ic_mean, "ic_std": ic_std})

mask = ~np.isnan(oof["lgbm"])        # rows covered by at least one fold
OOF = pd.DataFrame({f: v[mask] for f, v in oof.items()})
Y_OOF = y.to_numpy()[mask]
OOF_ROWS = train_df.loc[mask]

display(pd.DataFrame(family_stats).round(4))
print(f"\nOOF rows: {len(OOF)}")

# Error diversity is the whole premise of the ensemble: the closer these
# correlations are to 1, the less there is to gain from blending.
print("\nprediction correlation between families:")
display(OOF.corr().round(3))

## 5. Blend weights and allocation scale

Two things are searched jointly on the out-of-fold rows:

* `w_lgbm, w_cat, w_ridge` — normalised to sum to one, so only two degrees of
  freedom are real;
* `k` — the single scalar in the naive allocation rule.

The objective is the **modified Sharpe of the resulting allocation**, not
Spearman. At this point the ranking is already fixed by the trained models; what
remains is how aggressively to convert it into a position, and that is a
risk-metric question.

In [ ]:
FWD_OOF = OOF_ROWS["forward_returns"].to_numpy()
RF_OOF = OOF_ROWS["risk_free_rate"].to_numpy()


def blend(preds: pd.DataFrame, weights: dict) -> np.ndarray:
    total = sum(weights.values())
    return sum(preds[f].to_numpy() * w for f, w in weights.items()) / total


def blend_objective(trial):
    weights = {f: trial.suggest_float(f"w_{f}", 0.0, 1.0)
               for f in ("lgbm", "catboost", "ridge")}
    if sum(weights.values()) < 1e-6:
        return -1e9
    k = trial.suggest_float("k", 1.0, 500.0, log=True)
    pos = naive_allocation(blend(OOF, weights), k=k)
    return modified_sharpe(pos, FWD_OOF, RF_OOF)


blend_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    study_name="stage3_blend",
)
blend_study.optimize(blend_objective, n_trials=N_TRIALS_BLEND,
                     show_progress_bar=False)

bp = blend_study.best_params
_raw = {f: bp[f"w_{f}"] for f in ("lgbm", "catboost", "ridge")}
_tot = sum(_raw.values())
WEIGHTS = {f: v / _tot for f, v in _raw.items()}
K = bp["k"]

print("blend weights:", {f: round(v, 3) for f, v in WEIGHTS.items()})
print(f"k = {K:.1f}")
print(f"OOF modified Sharpe: {blend_study.best_value:.4f}")

### Is `k` overfitted?

A single scalar fitted on several hundred OOF rows should have a broad optimum.
If the curve below is flat across a wide range, the choice is safe; if it spikes,
the allocation is being tuned to noise and `k` should be pinned to a round number
instead.

In [ ]:
blended_oof = blend(OOF, WEIGHTS)
grid = [1, 2, 5, 10, 20, 50, 100, 200, 500]
sens = pd.DataFrame([{
    "k": g,
    "modified_sharpe": modified_sharpe(naive_allocation(blended_oof, k=g),
                                       FWD_OOF, RF_OOF),
    "mean_weight": float(np.mean(naive_allocation(blended_oof, k=g))),
} for g in grid]).round(4)
display(sens)

print("passive w=1 reference:",
      round(modified_sharpe(np.ones(len(FWD_OOF)), FWD_OOF, RF_OOF), 4))

## 6. Cross-validated evaluation of the ensemble

Scored per fold on the shared splits, using the blend weights and `k` chosen
above, so these numbers line up with every other notebook's CV block.

In [ ]:
fold_metrics = []
for fold in folds:
    idx = fold.val_idx
    val = train_df.iloc[idx]
    preds = pd.DataFrame({f: oof[f][idx] for f in WEIGHTS})
    pred = blend(preds, WEIGHTS)
    pos = naive_allocation(pred, k=K)

    m = evaluate(y.iloc[idx].to_numpy(), pred, weights=pos,
                 forward_returns=val["forward_returns"].to_numpy(),
                 risk_free_rate=val["risk_free_rate"].to_numpy())
    m["fold"] = fold.index
    fold_metrics.append(m)

display(pd.DataFrame(fold_metrics)[
    ["fold", "spearman_ic", "rmse", "hit_rate", "modified_sharpe", "sharpe",
     "vol_ratio", "benchmark_sharpe", "mean_weight"]].round(4))

cv_metrics = aggregate_folds([{k_: v for k_, v in m.items() if k_ != "fold"}
                              for m in fold_metrics])
print("\nmean Spearman IC:      {spearman_ic_mean:+.4f} (sd {spearman_ic_std:.4f})"
      .format(**cv_metrics))
print("mean modified Sharpe:  {modified_sharpe_mean:+.4f} (sd {modified_sharpe_std:.4f})"
      .format(**cv_metrics))
print("benchmark:             {benchmark_sharpe_mean:+.4f}".format(**cv_metrics))

### Where the gain came from

Two attributions on identical folds: the ensemble against each family alone, and
the allocation rule against the alternatives. Separating them matters — if the
whole improvement is the allocation rule, the ensemble is not pulling its weight
and the simpler Stage 2 model with a better sizer would be the honest
recommendation.

In [ ]:
rows = []
for label, pred_vec in [
    ("lgbm alone", OOF["lgbm"].to_numpy()),
    ("catboost alone", OOF["catboost"].to_numpy()),
    ("ridge alone", OOF["ridge"].to_numpy()),
    ("equal-weight blend", blend(OOF, {f: 1.0 for f in WEIGHTS})),
    ("fixed 0.30/0.35/0.35 (100th)",
     blend(OOF, {"ridge": 0.30, "lgbm": 0.35, "catboost": 0.35})),
    ("tuned blend", blended_oof),
]:
    rows.append({
        "variant": label,
        "spearman_ic": spearman_ic(Y_OOF, pred_vec),
        "modified_sharpe": modified_sharpe(
            naive_allocation(pred_vec, k=K), FWD_OOF, RF_OOF),
    })
display(pd.DataFrame(rows).round(4))

In [ ]:
rows = []
for label, pos in [
    ("naive (this stage)", naive_allocation(blended_oof, k=K)),
    ("binary (Stage 2 / 61st)", binary_allocation(blended_oof)),
    ("vol-target (4th)", vol_target_allocation(
        blended_oof, OOF_ROWS["vol_20"].to_numpy(), target_vol=0.12, k=K)),
    ("naive + smoothing", smooth_weights(naive_allocation(blended_oof, k=K))),
    ("passive w=1", np.ones(len(blended_oof))),
]:
    r = modified_sharpe(pos, FWD_OOF, RF_OOF, return_components=True)
    r["rule"] = label
    r["mean_weight"] = float(np.mean(pos))
    r["turnover"] = float(np.mean(np.abs(np.diff(pos))))
    rows.append(r)
display(pd.DataFrame(rows)[["rule", "modified_sharpe", "sharpe", "vol_ratio",
                            "vol_penalty", "return_penalty", "mean_weight",
                            "turnover"]].round(4))

## 7. Held-out public block

Every family refit on the whole training period, blended with the chosen
weights, sized with the chosen `k`, and scored once on the 180 `date_id`s that
were never fitted on or selected against.

In [ ]:
final_models, pub_preds = {}, {}

m = lgb.LGBMRegressor(**{**BEST["lgbm"], "random_state": SEED, "verbosity": -1})
m.fit(X, y)
pub_preds["lgbm"] = m.predict(public_df[FEATURES])
final_models["lgbm"] = m

m = CatBoostRegressor(**{**BEST["catboost"], "random_seed": SEED,
                         "verbose": 0, "loss_function": "RMSE"})
m.fit(X, y)
pub_preds["catboost"] = m.predict(public_df[FEATURES])
final_models["catboost"] = m

scaler = StandardScaler().fit(X)
m = Ridge(alpha=BEST["ridge"]["alpha"], random_state=SEED)
m.fit(scaler.transform(X), y)
pub_preds["ridge"] = m.predict(scaler.transform(public_df[FEATURES]))
final_models["ridge"] = (scaler, m)

pub_pred = blend(pd.DataFrame(pub_preds), WEIGHTS)
pub_pos = naive_allocation(pub_pred, k=K)

public_metrics = evaluate(
    public_df[TARGET].to_numpy(), pub_pred, weights=pub_pos,
    forward_returns=public_df["forward_returns"].to_numpy(),
    risk_free_rate=public_df["risk_free_rate"].to_numpy(),
)
print({k_: round(v, 4) for k_, v in public_metrics.items() if isinstance(v, float)})

## 8. Comparison against every previous model

All rows come from `results/leaderboard.csv`, written by the Stage 1 and Stage 2
notebooks on the same folds and the same held-out block.

In [ ]:
MODEL_NAME = "improved_ensemble"

save_result(model=MODEL_NAME, stage="improved", metrics=cv_metrics, split="cv",
            params={"weights": WEIGHTS, "k": K, **BEST},
            notes="LightGBM + CatBoost + Ridge, Optuna on HPs and blend weights, "
                  "naive allocation")
save_result(model=MODEL_NAME, stage="improved", metrics=public_metrics,
            split="public", params={"weights": WEIGHTS, "k": K},
            notes="refit on full train period, scored once on held-out 180 rows")

save_predictions(MODEL_NAME, OOF_ROWS[DATE_COL], Y_OOF, blended_oof,
                 naive_allocation(blended_oof, k=K))

print("=== cross-validated (identical purged folds) ===")
display(compare(split="cv"))
print("=== held-out public block (180 rows, scored once) ===")
display(compare(split="public"))

## Conclusions

**Read the attribution tables in section 6 before the headline.** The two
questions that matter are whether the ensemble beat its best single member, and
whether the allocation rule or the model produced the improvement. If the tuned
blend barely beats the equal-weight blend, the weight search is noise and the
fixed weights from the 100th place write-up are the more defensible choice; if
the naive rule accounts for nearly all the gain over Stage 2, then the honest
conclusion is the one both the 4th and 100th place authors reached
independently — that position sizing under this metric matters more than the
forecast.

**On the metric.** `modified_sharpe` is our re-implementation of the competition
metric (volatility cliff at 120% of market volatility, quadratic penalty for
underperforming buy-and-hold), not the organisers' code. Every model in the
comparison is scored with it on identical folds, so the ranking between them is
meaningful; the absolute values are not comparable to published leaderboard
scores.

**Residual risk.** Hyperparameters, blend weights and `k` were all selected
using the same four folds. The out-of-fold construction and the coarse, flat `k`
grid limit the damage, but the cross-validated numbers are still optimistic
relative to what a genuinely unseen period would produce — which is exactly the
gap the 61st place author described after watching a well-validated model meet
live data. The single-shot public block is the closer estimate, and it is one
sample of 180 rows.